# 充换电订单逐小时摊分
输入应覆盖目标自然年。确认目标年内无订单的小时确实无交易，且订单已去重并剔除冲正。

In [ ]:
from load_common import export, hourly_frame, fill_meter_gaps, PERIODS
import numpy as np
import pandas as pd

YEAR = 2026
TARIFF = "inputs/tariff.xlsx"
INPUT = "inputs/orders.xlsx"
OUTPUT = "outputs/orders_8760.xlsx"
SHEET = 0
START_COL = "充换电开始时间"
END_COL = "充换电结束时间"
ENERGY_COL = "充换电量"


In [ ]:
raw = pd.read_excel(INPUT, sheet_name=SHEET)
start = pd.to_datetime(raw[START_COL], errors="coerce")
end = pd.to_datetime(raw[END_COL], errors="coerce")
energy = pd.to_numeric(raw[ENERGY_COL], errors="coerce")
valid = start.notna() & (end > start) & np.isfinite(energy) & (energy > 0)
print("剔除无效订单数：", (~valid).sum())
frame = hourly_frame(YEAR, TARIFF)
year_start = frame.index[0]
year_end = frame.index[-1] + pd.Timedelta(hours=1)
hour = pd.Timedelta(hours=1)
load = pd.Series(0.0, index=frame.index)
expected = 0.0
for s, e, kwh in zip(start[valid], end[valid], energy[valid]):
    left, right = max(s, year_start), min(e, year_end)
    if left >= right:
        continue
    expected += kwh * ((right - left) / (e - s))
    for bucket in pd.date_range(left.floor("h"), right.ceil("h"), freq="h", inclusive="left"):
        overlap = min(right, bucket + hour) - max(left, bucket)
        load.loc[bucket] += kwh * (overlap / (e - s))

if expected <= 0:
    raise ValueError("目标年无有效交易，请核对年份和订单覆盖范围")
if not np.isclose(load.sum(), expected):
    raise ValueError("逐时电量与目标年内订单电量不一致")
frame["负载"] = load
frame["负载来源"] = np.where(load > 0, "订单摊分", "无订单")
audit = pd.DataFrame({"目标年内订单电量_kWh": [expected], "逐时汇总电量_kWh": [load.sum()]})
export(frame, OUTPUT, audit)
